# Radio Bulletin Generator — Three Languages

**WeatherSpeak PH** — Gemma 4 Hackathon

## Objective

Generate natural, radio-ready weather bulletins in **three languages**:
1. **English** (Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

- **Target length**: ~750 words (~5 minutes at 150 wpm radio reading pace)
- **Style**: Philippine radio broadcast — authoritative, clear, flowing prose
- **Model**: Gemma 4 E4B via Ollama (text-only — no vision needed here)
- **Input**: Full markdown bulletin text (from notebook 04)
- **Output**: Plain text scripts saved to `data/radio_bulletins/`

### Two sample bulletins
1. `PAGASA_20-19W_Pepito_SWB#01` — Severe Weather Bulletin, Tropical Depression Pepito
2. `PAGASA_22-TC02_Basyang_TCA#01` — Tropical Cyclone Alert, Basyang

**Total output**: 6 scripts (2 bulletins × 3 languages)

In [3]:
import json
import requests
import time
from pathlib import Path

OLLAMA_API = "http://localhost:11434"
MODEL_NAME = "gemma4:e4b"
TIMEOUT = 10 * 60  # 10 minutes

data_dir = Path("../data")
markdown_dir = data_dir / "gemma4_results"
output_dir = data_dir / "radio_bulletins"
output_dir.mkdir(exist_ok=True)

# The two bulletins we are working with
SAMPLES = [
    "PAGASA_20-19W_Pepito_SWB#01",
    "PAGASA_22-TC02_Basyang_TCA#01",
]

# Verify Ollama is running
try:
    r = requests.get(f"{OLLAMA_API}/api/tags", timeout=5)
    names = [m['name'] for m in r.json()['models']]
    status = "\u2713" if any(MODEL_NAME in n for n in names) else "\u26a0\ufe0f  model not found"
    print(f"\u2713 Ollama running \u2014 {MODEL_NAME}: {status}")
except Exception as e:
    print(f"\u2717 Ollama not reachable: {e}")

print(f"\u2713 Output dir: {output_dir.absolute()}")


✓ Ollama running — gemma4:e4b: ✓
✓ Output dir: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins


## 1. Load Bulletin Markdown

Load the raw markdown text extracted by notebook 04. This is the primary input — no structured JSON used.

In [4]:
def load_bulletin(stem: str) -> dict:
    """Load the raw markdown for a bulletin stem (primary input for generation)."""
    markdown_path = markdown_dir / f"{stem}_markdown.md"

    if not markdown_path.exists():
        raise FileNotFoundError(f"Markdown file not found: {markdown_path}")

    markdown = markdown_path.read_text(encoding="utf-8")

    return {"stem": stem, "markdown": markdown}


bulletins = [load_bulletin(s) for s in SAMPLES]

for b in bulletins:
    print(f"\n{b['stem']}")
    print(f"  Markdown: {len(b['markdown'])} chars")
    print(f"  Preview:  {b['markdown'][:120].strip()!r}")



PAGASA_20-19W_Pepito_SWB#01
  Markdown: 4825 chars
  Preview:  '## REPUBLIC OF THE PHILIPPINES\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PHILIPPINE ASTROPHYSICAL, GEOPHYSICAL AND ASTR'

PAGASA_22-TC02_Basyang_TCA#01
  Markdown: 3224 chars
  Preview:  '# PHILIPPINE ARCHIPELAGO\n**DEPARTMENT OF SCIENCE AND TECHNOLOGY**\n**PAGASA**\nPhilippine Atmospheric, Geophysical and Ast'


## 2. Radio Bulletin Prompts — Three Languages

Generate radio bulletins in:
1. **English** (standard Philippine radio English)
2. **Tagalog** (Filipino, standard broadcast Tagalog)
3. **Cebuano** (standard written Cebuano/Bisaya)

All versions maintain the same style:
- Flowing prose — no bullet points, no tables, no markdown

- ~750 words (~5 minutes reading time)- Clear structure: opening → current situation → forecast → affected areas → public safety → closing

- Repeat storm name and signal areas (radio listeners may tune in mid-broadcast)- Plain spoken numbers

In [5]:
PROMPTS = {
    "en": {
        "system": """You are a Philippine radio weather broadcaster writing a spoken bulletin script for on-air reading in English.

STYLE RULES:
- Write in flowing, natural prose suitable for reading aloud.
- Use a calm, authoritative tone appropriate for public emergency broadcasts.
- Spell out numbers when they will be read aloud (e.g. "one hundred thirty kilometres per hour", "Signal Number Two").
- Repeat the storm name and the most critical warning signal at least twice — radio listeners may tune in partway through.
- Use natural spoken transitions: "At this time...", "Moving on to the forecast...", "Residents are urged to..."
- Reference the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA, by name at the start.
- Close with the time of the next scheduled bulletin update.

FORMATTING: Output in Markdown so the script is easy to read and review.
- Use a top-level heading for the bulletin title (storm name + bulletin type)
- Use second-level headings for each section: Current Situation, Forecast Track, Affected Areas, Public Safety Advisory, Closing
- Bold the storm name and signal numbers on first mention in each section
- Keep the prose itself natural and speakable — the markdown is for readability only

LENGTH: Target approximately 750 words — this should read aloud in about five minutes at a steady broadcast pace.""",

        "user_template": (
            "Write a five-minute radio broadcast bulletin script in English "
            "based on the following PAGASA weather bulletin text.\n\n"
            "{markdown}\n\n"
            "Write the radio script now. Use Markdown formatting, approximately 750 words."
        ),
    },

    "tl": {
        "system": """Ikaw ay isang Filipino radio weather broadcaster na sumusulat ng spoken bulletin script para basahin sa ere sa Tagalog.

MGA PATAKARAN SA ESTILO:
- Sumulat sa natural na daloy ng prosa na angkop para basahin nang malakas.
- Gumamit ng kalmado at may awtoridad na tono na angkop para sa public emergency broadcasts.
- I-spell out ang mga numero kapag babasahin (halimbawa: "isandaan at tatlumpung kilometro bawat oras", "Signal Number Two").
- Ulitin ang pangalan ng bagyo at ang pinaka-kritikal na babala nang kahit dalawang beses.
- Gumamit ng natural na transisyon: "Sa ngayon...", "Patungo sa forecast...", "Hinihikayat ang mga residente na..."
- Banggitin ang PAGASA sa simula ng bulletin.
- Magtapos sa oras ng susunod na scheduled bulletin update.

FORMATTING: Mag-output sa Markdown para madaling basahin at suriin.
- Gumamit ng top-level heading para sa pamagat ng bulletin (pangalan ng bagyo + uri ng bulletin)
- Gumamit ng second-level headings para sa bawat seksyon: Kasalukuyang Sitwasyon, Forecast Track, Mga Apektadong Lugar, Payo sa Kaligtasan, Pangwakas
- I-bold ang pangalan ng bagyo at signal numbers sa unang pagbanggit sa bawat seksyon
- Panatilihing natural ang prosa para mabasa nang maayos

HABA: Target na humigit-kumulang 750 salita — dapat itong mabasa sa loob ng limang minuto.""",

        "user_template": (
            "Sumulat ng limang minutong radio broadcast bulletin script sa Tagalog "
            "batay sa sumusunod na PAGASA weather bulletin text.\n\n"
            "{markdown}\n\n"
            "Sumulat ng radio script ngayon. Gumamit ng Markdown formatting, humigit-kumulang 750 salita."
        ),
    },

    "ceb": {
        "system": """Ikaw usa ka Filipino radio weather broadcaster nga direkta nakigsulti sa mga mamumuno sa Cebuano. Isulat ang bulletin sama sa imong gisulti sa radyo — natural, conversational, ug dali masabtan.

MGA LAGDA SA ESTILO (CONVERSATIONAL RADIO CEBUANO):
- Pagsulat sama sa regular nga radyo broadcaster nga nakigsulti sa mga tagapaminaw — dili formal, natural lang nga Bisaya.
- Gamita ang mga common nga radio phrases: "Ato pang hisgutan...", "Karon, mga higala...", "Unya ha...", "Importante nga..."
- I-spell out ang mga numero sama sa pagsulti (pananglitan: "usa ka gatos ug katluan ka kilometro matag oras", "Signal Number Two").
- Sublia ang ngalan sa bagyo ug signal areas og duha o tulo ka beses — basin lang dili makahibalo ang uban.
- Gamita ang conversational nga transitions: "Karon ha...", "Unya kini...", "Importante kaayo nga...", "Mga kauban..."
- Mention PAGASA sa sinugdan pero dili kaayo formal — natural lang.
- Tapuson sama sa regular nga radyo: "Tan-awa nato pag-usab sa..." o "Magkita tag usab sa..."

FORMATTING: Mag-output sa Markdown aron dali mabasahan.
- Paggamit og top-level heading alang sa titulo sa bulletin (ngalan sa bagyo + matang sa bulletin)
- Paggamit og second-level headings alang sa matag seksyon pero simple lang: Unsay Nahitabo Karon, Asa Padulong ang Bagyo, Kinsa ang Apektado, Unsa ang Kinahanglan Buhaton, Pangwakas
- I-bold ang ngalan sa bagyo ug signal numbers
- Pero ang script mismo kinahanglan conversational — parang nag-istambay lang mo sa radyo

GITAS-ON: Mga 750 ka pulong — lima ka minuto nga radyo talk, dili basahon nga formal.""",

        "user_template": (
            "Isulat ang lima ka minutong conversational radio bulletin sa Cebuano "
            "base sa PAGASA weather bulletin text nga naay sulod dinhi.\n\n"
            "{markdown}\n\n"
            "Pagsulat karon — conversational Bisaya, parang nag-istorya lang sa radyo. Markdown format, mga 750 ka pulong."
        ),
    },
}


def build_user_prompt(bulletin: dict, language: str) -> str:
    """Build the user prompt using the full markdown bulletin text as input."""
    template = PROMPTS[language]["user_template"]
    return template.format(markdown=bulletin["markdown"])


print("\u2713 Prompts defined for 3 languages")
for lang, prompts in PROMPTS.items():
    print(f"  {lang}: system={len(prompts['system'])} chars, template={len(prompts['user_template'])} chars")


✓ Prompts defined for 3 languages
  en: system=1362 chars, template=206 chars
  tl: system=1304 chars, template=227 chars
  ceb: system=1579 chars, template=250 chars


## 3. Generate Radio Bulletins — All Three Languages

Call Gemma 4 for each bulletin in English, Tagalog, and Cebuano (6 total scripts).

In [6]:
def generate_radio_bulletin(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a radio broadcast script for one bulletin in the specified language."""
    stem = bulletin["stem"]
    lang_names = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}
    print(f"\nGenerating: {stem} ({lang_names[language]})")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": PROMPTS[language]["system"]},
            {"role": "user", "content": build_user_prompt(bulletin, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    elapsed = time.time() - t_start

    script = response.json().get("message", {}).get("content", "").strip()

    # Word count (strip markdown syntax for accurate spoken-word estimate)
    import re
    plain = re.sub(r"[#*_`>\-]", "", script)
    word_count = len(plain.split())
    reading_minutes = word_count / 150  # ~150 wpm for radio

    # Save as markdown file
    out_path = output_dir / f"{stem}_radio_{language}.md"
    out_path.write_text(script, encoding="utf-8")

    print(f"  \u2713 Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}  |  Est. reading time: {reading_minutes:.1f} min")
    print(f"  Saved \u2192 {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "script": script,
        "word_count": word_count,
        "reading_minutes": reading_minutes,
        "elapsed": elapsed,
    }


results = []
for bulletin in bulletins:
    for lang in ["en", "tl", "ceb"]:
        result = generate_radio_bulletin(bulletin, lang)
        results.append(result)

print(f"\n\u2713 Done \u2014 {len(results)} scripts generated ({len(bulletins)} bulletins \u00d7 3 languages)")



Generating: PAGASA_20-19W_Pepito_SWB#01 (English)
  ✓ Generated in 47.6s
  Words: 676  |  Est. reading time: 4.5 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_en.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Tagalog)
  ✓ Generated in 62.2s
  Words: 672  |  Est. reading time: 4.5 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_tl.md

Generating: PAGASA_20-19W_Pepito_SWB#01 (Cebuano)
  ✓ Generated in 61.8s
  Words: 644  |  Est. reading time: 4.3 min
  Saved → PAGASA_20-19W_Pepito_SWB#01_radio_ceb.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (English)
  ✓ Generated in 47.4s
  Words: 654  |  Est. reading time: 4.4 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_en.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Tagalog)
  ✓ Generated in 62.1s
  Words: 684  |  Est. reading time: 4.6 min
  Saved → PAGASA_22-TC02_Basyang_TCA#01_radio_tl.md

Generating: PAGASA_22-TC02_Basyang_TCA#01 (Cebuano)
  ✓ Generated in 59.6s
  Words: 663  |  Est. reading time: 4.4 min
  Saved → PAGASA_22-TC02_Basyan

## 6. TTS Plain Text Generation — All Three Languages

Generate TTS-optimized plain text versions of all radio scripts via a second
Gemma4 prompt. These files feed directly into notebook 08 — no markdown stripping needed.

**Output:** `data/radio_bulletins/{stem}_tts_{lang}.txt` (CEB, TL, EN)

In [ ]:
TTS_PROMPTS = {
    "ceb": {
        "system": """Ikaw usa ka espesyalista sa Cebuano nga nagsulat og plain text nga angay para sa text-to-speech synthesis.

Ang imong trabaho:
- Basaha ang markdown radio script nga gihatag
- Isulat kini pag-usab isip natural nga flowing prose SA CEBUANO LAMANG — walay markdown
- WALA markdown: wala headings (#), wala bullet points (-), wala asterisks (*), wala bold/italic
- Para sa mga English proper nouns o teknikal nga termino, i-spell sila phonetically sa Cebuano:
  - PAGASA → pa-ga-sa
  - Northern Luzon → nor-dern lu-son
  - Signal Number One / Two / Three → sig-nal nam-ber wan / tu / tri
  - tropical depression → tro-pi-kal di-pre-syon
  - tropical storm → tro-pi-kal storm
  - kilometers per hour → ki-lo-me-tros sa usa ka oras
  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west
- Pahimusa ang paragraph structure: blank lines tali sa mga paragraph
- AYAW pagdugang og bisan unsa nga texto nga wala sa orihinal nga script
- Output: plain text lamang, walay bisan unsang markup o formatting characters""",

        "user_template": (
            "Basaha kining markdown radio script ug isulat kini pag-usab isip TTS-ready plain Cebuano text.\n\n"
            "{markdown}\n\n"
            "Isulat ang plain Cebuano text karon. Cebuano nga pulong lamang, phonetically spelled kung "
            "kinahanglan, paragraph breaks (blank lines) para sa natural nga pausing. Walay markdown."
        ),
    },
    "tl": {
        "system": """Ikaw ay isang espesyalista sa Tagalog na sumusulat ng plain text na angkop para sa text-to-speech synthesis.

Ang iyong trabaho:
- Basahin ang markdown radio script na ibinigay
- Isulat muli ito bilang natural na flowing prose SA TAGALOG LAMANG — walang markdown
- WALANG markdown: walang headings (#), walang bullet points (-), walang asterisks (*), walang bold/italic
- Para sa mga English proper nouns o teknikal na termino, i-spell ang mga ito nang phonetically sa Tagalog:
  - PAGASA → pa-ga-sa
  - Northern Luzon → nor-dern lu-son
  - Signal Number One / Two / Three → sig-nal nam-ber wan / tu / tri
  - tropical depression → tro-pi-kal di-pre-syon
  - tropical storm → tro-pi-kal storm
  - kilometers per hour → ki-lo-me-tro ba-wat o-ras
  - northeast / southeast / northwest / southwest → nor-ist / sow-ist / nor-west / sow-west
- Panatilihin ang paragraph structure: blank lines sa pagitan ng mga paragraph
- HUWAG magdagdag ng anumang texto na wala sa orihinal na script
- Output: plain text lamang, walang anumang markup o formatting characters""",

        "user_template": (
            "Basahin ang markdown radio script na ito at isulat muli ito bilang TTS-ready plain Tagalog text.\n\n"
            "{markdown}\n\n"
            "Isulat ang plain Tagalog text ngayon. Tagalog na salita lamang, phonetically spelled kung "
            "kinakailangan, paragraph breaks (blank lines) para sa natural na pausing. Walang markdown."
        ),
    },
    "en": {
        "system": """You are a specialist in writing plain text suitable for text-to-speech synthesis.

Your task:
- Read the provided markdown radio script
- Rewrite it as natural flowing prose IN ENGLISH ONLY — no markdown
- NO markdown: no headings (#), no bullet points (-), no asterisks (*), no bold/italic
- Keep technical terms clear and pronounceable
- Maintain paragraph structure: blank lines between paragraphs
- DO NOT add any text that wasn't in the original script
- Output: plain text only, no markup or formatting characters""",

        "user_template": (
            "Read this markdown radio script and rewrite it as TTS-ready plain English text.\n\n"
            "{markdown}\n\n"
            "Write the plain English text now. Clear pronunciation-friendly English, "
            "paragraph breaks (blank lines) for natural pausing. No markdown."
        ),
    },
}


def build_tts_prompt(markdown_script: str, language: str) -> str:
    """Build the user prompt for TTS text generation."""
    return TTS_PROMPTS[language]["user_template"].format(markdown=markdown_script)


print("✓ TTS_PROMPTS defined for CEB, TL, EN")
print(f"  ceb: system={len(TTS_PROMPTS['ceb']['system'])} chars")
print(f"  tl:  system={len(TTS_PROMPTS['tl']['system'])} chars")
print(f"  en:  system={len(TTS_PROMPTS['en']['system'])} chars")

✓ TTS_PROMPTS defined for CEB
  ceb: system=1040 chars


In [ ]:
def generate_tts_text(bulletin: dict, language: str) -> dict:
    """Call Gemma 4 to generate a TTS-ready plain text script for one bulletin."""
    stem = bulletin["stem"]
    lang_names = {"ceb": "Cebuano", "tl": "Tagalog", "en": "English"}
    print(f"\nGenerating TTS text: {stem} ({lang_names[language]})")

    markdown_script = (output_dir / f"{stem}_radio_{language}.md").read_text(encoding="utf-8")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": TTS_PROMPTS[language]["system"]},
            {"role": "user", "content": build_tts_prompt(markdown_script, language)},
        ],
        "stream": False,
    }

    t_start = time.time()
    response = requests.post(f"{OLLAMA_API}/api/chat", json=payload, timeout=TIMEOUT)
    response.raise_for_status()
    elapsed = time.time() - t_start

    tts_text = response.json().get("message", {}).get("content", "").strip()

    out_path = output_dir / f"{stem}_tts_{language}.txt"
    out_path.write_text(tts_text, encoding="utf-8")

    word_count = len(tts_text.split())
    print(f"  ✓ Generated in {elapsed:.1f}s")
    print(f"  Words: {word_count}")
    print(f"  Saved → {out_path.name}")

    return {
        "stem": stem,
        "language": language,
        "tts_text": tts_text,
        "word_count": word_count,
        "elapsed": elapsed,
    }


# Generate TTS text for both bulletins — all three languages
tts_results = []
for bulletin in bulletins:
    for lang in ["ceb", "tl", "en"]:
        result = generate_tts_text(bulletin, lang)
        tts_results.append(result)

print(f"\n✓ Done — {len(tts_results)} TTS text files generated")

# Preview first bulletin (CEB)
print("\nPREVIEW — CEB TTS text (first 500 chars)")
print("=" * 60)
ceb_result = [r for r in tts_results if r["language"] == "ceb"][0]
print(ceb_result["tts_text"][:500])
print("...")


Generating TTS text: PAGASA_20-19W_Pepito_SWB#01 (Cebuano)
  ✓ Generated in 49.6s
  Words: 606
  Saved → PAGASA_20-19W_Pepito_SWB#01_tts_ceb.txt

Generating TTS text: PAGASA_22-TC02_Basyang_TCA#01 (Cebuano)
  ✓ Generated in 45.2s
  Words: 599
  Saved → PAGASA_22-TC02_Basyang_TCA#01_tts_ceb.txt

✓ Done — 2 TTS text files generated

PREVIEW — CEB TTS text (first 500 chars)
Maayong buntag, mga kaigsoonan! Happy Sunday, ug kanunayng pag-amping kanunay, okay? Karon, mga higala, wala lang na ta’y paghisgot og pagka-relax lang sa atong weather. Kasagaran namong himoon ni, pero kinahanglan gyud nako nga i-update mo og importante kaayo nga weather bulletin gikan sa atong kaigsuon sa pa-ga-sa.

Tungko mi nga kanunay'g pagbantay ninyo sa kalawasan, ug aron kana magmalipayon, ang pagbantay sa panahon gyud ang pinaka-importante.

Karon ha, kinahanglan nako nga atong hisgutan
...


## 4. Review Output

Display each generated script for manual inspection (grouped by bulletin, showing all languages).

In [9]:
from IPython.display import display, Markdown
from collections import defaultdict

grouped = defaultdict(list)
for r in results:
    grouped[r["stem"]].append(r)

lang_labels = {"en": "English", "tl": "Tagalog", "ceb": "Cebuano"}

for stem, versions in grouped.items():
    display(Markdown(f"---\n# {stem}\n---"))
    for version in versions:
        lang = version["language"]
        meta = (
            f"**Language:** {lang_labels[lang]}  \u2502  "
            f"**Words:** {version['word_count']}  \u2502  "
            f"**Est. reading time:** {version['reading_minutes']:.1f} min"
        )
        display(Markdown(f"## {lang_labels[lang]} Version\n\n{meta}\n\n---\n\n{version['script']}"))


---
# PAGASA_20-19W_Pepito_SWB#01
---

## English Version

**Language:** English  │  **Words:** 676  │  **Est. reading time:** 4.5 min

---

# Tropical Depression "Pepito": Severe Weather Advisory Bulletin

## Current Situation

Good morning. This is a special severe weather advisory bulletin broadcast by the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA. We are monitoring the situation concerning Tropical Depression **Pepito**.

At this time, **Pepito** is located in the eastern portion of our region, and PAGASA confirms that this system poses a significant threat. We are issuing this alert because the Low Pressure Area that developed east of Catanduanes has organized into Tropical Depression **Pepito**. Listeners, please remain vigilant.

As of this morning, PAGASA advises that **Pepito** has the potential to further intensify, and the entire weather pattern remains dynamic. We advise all residents to keep abreast of the latest advisories, as the system has a potential to strengthen into a Tropical Storm or even a full-blown typhoon.

## Forecast Track

Moving on to the forecast track, we will outline the anticipated movement of **Pepito**.

As of the time of this bulletin, the center of Tropical Depression **Pepito** was estimated to be passing approximately eight hundred twenty kilometers east of Virday, Catanduanes.

Looking ahead, we project the following trajectory. Within the next twenty-four hours, **Pepito** is expected to shift toward six hundred kilometers east of Cagayan, Aurora. By forty-eight hours from now, the system should approach an area east of Pangasinan, Isabela.

Further out, by seventy-two hours—Thursday morning—the storm is forecasted to be about two hundred fifty-five kilometers west of Pangasinan City. This westward movement continues, bringing us to the ninety-six-hour mark on Friday morning, where **Pepito** will be near one hundred fifty kilometers west of Pangasinan City. The system is expected to continue tracking westward, moving over the open waters, with a final forecast point at the ninety-six-hour mark on Saturday morning, located eight hundred seventy kilometers west of Northern Luzon.

The current sustained winds near the center are reported at up to forty-five kilometers per hour, with gusts reaching as high as fifty-five kilometers per hour.

## Affected Areas

Residents are urged to pay close attention to the geographical areas that will be impacted by **Pepito**.

The areas expected to experience the most severe impact are Catanduanes, Northern Samar, Samar, Southern Leyte, Masbate, Albay, Camarines Norte, Camarines Sur, Quezon, and parts of Quezon. We must emphasize that the coastal areas of Northern Luzon, including Isabela, Cagayan, Ilocos Norte, Aurora, Zambales, Bataan, and parts of Pangasinan, will also be significantly affected.

For those near the Bicol Region and Catanduanes, strong winds and heavy rainfall are expected throughout the day and into the night. This **Tropical Depression Pepito** is bringing rainfall potential across the entire Visayas region, particularly during the night hours.

## Public Safety Advisory

Moving on to the public safety advisory, the primary hazards to anticipate are heavy rainfall and high seas.

For all coastal waters, we are forecasting moderate to strong seas, reaching up to four point five meters, particularly on the northeast side. Fisherfolk and those utilizing small sea craft or commercial vessels are strongly advised to seek safe harbor immediately, as rough waters are expected in the vicinity of the storm.

**Pepito** is a significant weather event. We repeat: Tropical Depression **Pepito** is bringing rainfall and strong winds, especially impacting the coastal regions of Northern Luzon and the islands of Samar and Catanduanes.

Residents in all listed affected areas must prepare for potential flood-prone conditions. PAGASA strongly advises the public to review family emergency kits, secure loose objects outdoors, and remain tuned to local government advisories. Do not travel unless absolutely necessary, and if you must travel, be prepared for disruptions.

## Closing

We remind everyone that while there is currently no tropical cyclone wind signal in effect, the threat posed by Tropical Depression **Pepito** requires the highest level of caution and preparedness from all sectors.

For the most up-to-date information regarding the track and intensity of **Pepito**, please continue to monitor official channels.

This concludes our current severe weather bulletin. We will provide the next comprehensive update on this developing situation at eleven o'clock in the morning. Stay safe, and please listen to official advisories.

## Tagalog Version

**Language:** Tagalog  │  **Words:** 672  │  **Est. reading time:** 4.5 min

---

# TROPICAL DEPRESSION "PEPITO" SEVERE WEATHER BULLETIN: PAGASA UPDATE

*(SOUND EFFECT: Short, authoritative, calming musical stinger)*

**BROADCASTER:** Magandang umaga po sa lahat ng ating nakikinig. Ito po ay isang espesyal na paghahatid mula sa *Philippine Atmospheric, Geophysical and Astronomical Service*, o PAGASA. Ang ating layunin ngayon ay magbigay ng pinakabagong kaalaman at pag-iingat patungkol sa pagdaan ng bagyo na kasalukuyang tinatawag na **Tropical Depression "PEPITO"**.

Hinihikayat po namin ang lahat ng mamamayan na manatiling kalmado, ngunit maging alerto at handa sa anumang pagbabago sa panahon. Pangalawang beses po nating paalalahanan ang lahat, ito ay tungkol sa **Tropical Depression "PEPITO"**.

***

## Kasalukuyang Sitwasyon

**BROADCASTER:** Sa pag-uumpisa ng bulletin na ito ngayong umaga, ang pag-amin po ng PAGASA ay nakumpirma na ang isang mababang *pressure area* na matatagpuan sa Silangang bahagi ng Catanduanes ay nag-develop na at napagtibay na bilang **Tropical Depression "PEPITO"**.

Sa ngayon, patuloy po itong sinusubaybayan nang mabuti ng mga eksperto. Ayon sa pagtatasa, may posibilidad po na lalo pang lumakas ang **Tropical Depression "PEPITO"** at maging isang Tropical Storm, o baka mas malala pa.

Ang direksyon ng paggalaw nito ay inaasahang magiging pang-kanluran hanggang pang-northwest, at pagkatapos, liliko ito patungo sa mas pang-kanluran, diretso patungo sa mga bahagi ng Northern Luzon.

Ang pangunahing babala po ngayong umaga ay may matitinding pagbuhos ng ulan at malalakas na hangin ang maaasahan sa mga coastal at lowland na lugar.

***

## Forecast Track at Ruta ng Bagyo

**BROADCASTER:** Patungo po tayo sa mga susunod na pagtataya sa ruta ng bagyo. Ang **Tropical Depression "PEPITO"** ay tinatarget na mag-landfall sa silangang baybayin ng Northern Luzon sa pagdating ng hapon ng ika-dalawampu't isang araw.

Tandaan po natin na ang pag-iingat ay hindi maaaring ipagpaliban. Ang paglalakbay o pag-alis sa mga lugar na inaasahang tamaan ay dapat isagawa nang maaga at may pag-iingat.

Base sa pagtataya ng PAGASA, ito ang mga inaasahang paglipat ng bagyo sa mga susunod na araw:

*   Sa loob ng dalawampung-kwatro oras, ang bagyo ay makakarating sa bahagi ng Cagayan, Aurora.
*   Sa loob ng apatnapu't walong oras, ang inaasahang posisyon nito ay magiging bahagi ng Pangasinan, Isabela.
*   At patuloy po itong lilipat patungo sa mas malayo at pang-kanluran na bahagi, na magpapatuloy sa pagtawid sa mga koral at tubig malayo sa mga pangunahing siyudad.

Ang paggalaw na ito ay nagpapakita na ang **Tropical Depression "PEPITO"** ay isang bagyong patuloy na gumagalaw, at kailangan natin ang patuloy nating paghahanda.

***

## Mga Apektadong Lugar at Panganib

**BROADCASTER:** Hinihimay po natin ang mga lugar na pinakamahihirap tamaan, o *most severely affected*.

Ang mga rehiyon na inaasahang magkakaroon ng pinakamalaking epekto ay ang **Catanduanes**, **Northern Samar**, **Samar**, at ang malalaking bahagi ng **Northern Luzon**.

Para sa mga maritime at coastal communities, ang panganib po ay kasama ang malalakas na hangin at pagbuhos ng malalakas na ulan. Sa mga baybayin, inaasahan po ang mga alon na may taas mula dalawang-kalahating metro hanggang apat-na-kalimang metro, na nangangahulugan ng malalaking panganib sa paglalayag at sa mga komunidad na malapit sa dagat.

Ang mga lugar na kailangan pong maging handa ay kinabibilangan ng:
*   Ang buong Bicol Region;
*   Ang Catanduanes;
*   Ang Northern Samar, Samar, Southern Leyte, Masbate, Albay, at iba pang kalapit na mga probinsya ng Quezon.
*   At hindi rin dapat kalimutan ang mga coastal areas sa Northern Luzon, kabilang ang Isabela, Cagayan, at Ilocos Norte.

***

## Payo sa Kaligtasan at Paghahanda

**BROADCASTER:** Dahil walang Tropical Cyclone Wind Signal na kasalukuyang ipinapailalim, ito ay nagpapatunay na kailangan nating manatiling mapagmatyag.

**Hinihikayat po namin ang mga residente na:**

Una, maging handa sa posibleng pagbaha at pagputol ng kuryente.
Pangalawa, iwasan po ang paglalakbay sa mga baybayin at ilog na umaapaw.
Pangatlo, ang mga sasakyang pandagat, kasama po ang mga mangingisda at *commercial vessel*, ay dapat maging labis na maingat at sumunod sa mga utos ng mga lokal na awtoridad.

Ang paghahanda po sa pagitan ng mga pamilya, pag-iimbak ng pagkain, at pagpaplano ng evacuation route ay mahalaga ngayon.

***

**Buod:** Ang patuloy na pagbabantay sa mga babala ng PAGASA ay kritikal. Manatiling nakatutok sa mga opisyal na anunsyo.

**Muli pong ipinapaalala:** Ang pangangalaga sa buhay ang pinakamahalaga.

*(Fade out music)*

## Cebuano Version

**Language:** Cebuano  │  **Words:** 644  │  **Est. reading time:** 4.3 min

---

# SEVERE WEATHER BULLETIN: TROPICAL DEPRESSION "PEPITO" UPDATE

**(Sound Effect: Short, gentle intro music fades out)**

**Broadcaster:** Maayong buntag, mga kaigsoonan! Happy Sunday, ug kanunayng pag-amping kanunay, okay? Karon, mga higala, wala lang na ta’y paghisgot og pagka-relax lang sa atong weather. Kasagaran namong himoon ni, pero kinahanglan gyud nako nga i-update mo og importante kaayo nga weather bulletin gikan sa atong kaigsuon sa PAGASA.

Tungko mi nga kanunay'g pagbantay ninyo sa kalawasan, ug aron kana magmalipayon, ang pagbantay sa panahon gyud ang pinaka-importante.

Karon ha, kinahanglan nako nga atong hisgutan ang usa ka tropikal nga depression nga gitawag og **Tropical Depression "PEPITO"**.

---

## Unsay Nahitabo Karon

**Broadcaster:** Mao ni, mga kauban, ang **Tropical Depression "PEPITO"** nahimo na pag-improve, o na-develop, didto sa parte sa dagat, sa sidlakan sa Catanduanes. Ayaw mo ka-alarm, pero mag-ampingay ta kaayo.

Ang PAGASA, sa ilang pag-forecast karon, nag-ingon nga ang **PEPITO** nagdala og potensyal nga motaas pa ang iyang intensity. Baka siya mo-transform pa’g Tropical Storm, o kaha, sa labing grabe, maabot pa gyud niya ang pagka-Typhoon. Mao ni ang pagka-andam natong tanan.

Unya, ang usa ka importanteng pahinumdom: Karon pa lang, wala pa'y signal nga gi-declare, pero ang mga hazard nga atong makabanggi, grabe kaayo na ka-severe.

---

## Asa Padulong ang Bagyo?

**Broadcaster:** Kanang mga pag-forecast sa direksyon, mga higala, ang **PEPITO** nagplano nga mu-move west-northwestward o west-northwestward. Mao ni ang initial nga direksyon.

Apan, atong pag-ayoan ang atong atensyon sa sunod nga mga adlaw, kasi makita ninyo ang mga lakang sa pag-uswag.

Sa usa ka kaadlaw (or tomorrow), ang target nga location niini kay **600 kilometers East of Cagayan, Aurora**. Mao ni ang mga mga lugar sa eastern coast sa Northern Luzon nga atong kinahanglan bantayan.

Ug sa dugay nga panahon, mga kauban, ang pag-forecast nagpakita nga ang **PEPITO** mag-abot sa eastern coast sa Northern Luzon sa hapon sa October ika-diyes-isan. Mao na ang panahon nga kinahanglan kitang kaayo nga andam.

---

## Kinsa ang Apektado?

**Broadcaster:** Karon ha, ang pinaka-importanteng bahin: Kinsa man ang makaka-feel aning ulan ug hangin nga bugkos-bugkos?

Mga kaigsoonan, ang mga lugar nga ina-abang sa **PEPITO** sa mga suod nga adlaw mao ang:

1.  **Sa Visayas Region:** Ilabi na sa mga lugar sama sa Catanduanes, Northern Samar, Samar, Southern Leyte, Masbate, Albay, Camarines Norte, ug Camarines Sur. Ang Visayas, labi na sa gabii, ang expected nga makadawat og kusog nga pag-ulan.
2.  **Sa Bicol at sa Eastern Coast:** Mga lugar nga nag-cover sa Quezon, ug sa mga coastal areas sa Northern Luzon, sama sa Isabela, Cagayan, ug Ilocos Norte.

Gikan sa pag-describe sa PAGASA, ang daghang pasidaan mao ang **moderate to strong seas**, mga upat ka-kuwarter hangtud sa lima ka-kuwarter ka metro! Busa, kung nag-sea travel mo, espesyal kaayo nga mag-amping mo.

---

## Unsa ang Kinahanglan Buhaton?

**Broadcaster:** Mao ni ang atong call to action, mga kauban. Walay kailangkit nga pag-ingon, pero ang pagka-andam dili ka maayo og sugod.

Una sa tanan: **I-monitor ang inyong balay.** Pag-andam og emergency kits. Ilisan na ang inyong mga emergency supplies. Tubig, non-perishable food, baterya nga spare.

Ikaduha: **Para sa mga nagka-travel sa tubig (Sea Travel):** Kung makita ninyo ang grabe nga hangin, mga dagat nga taas, o kusog nga ulan, palihug ayaw pag-travel. Hulat lang mo sa pag-clear sa panahon. Ang kinabuhi mas importante pa sa bisan unsang destination.

Ikatulo: **Paminawon ang mga opisyal nga paathag.** Kita magpabilin nga alerto. Ang PAGASA mo-release pa gyud og sunod nga bulletin sa alas onse sa buntag. Kinahanglan natong bantayan kana.

Tandaan ninyo, ang atong pagka-andam kini ang atong pinakadako nga gi-depensa batok sa kalikopan.

---

## Pangwakas

**Broadcaster:** Mga kaigsoonan, ang **Tropical Depression "PEPITO"** nagpadala kanato og seryoso kaayo nga mensahe: Pag-andam.

Ayaw kalimti nga ang kaayohan ninyo, ang inyong pamilya, ug ang inyong komunidad mao gyud ang atong prayoridad. Magkugi ta, mag-amping ta.

Kami nag-amping sa inyong tanan. Pag-amping kanunay! Mao ni si [Your Name/Station Name] nagpahibalo kaninyo. Hangtud sa sunod, kabay paayo!

---
# PAGASA_22-TC02_Basyang_TCA#01
---

## English Version

**Language:** English  │  **Words:** 654  │  **Est. reading time:** 4.4 min

---

# Tropical Storm Malakas Advisory: Weather Bulletin

## Current Situation

Good day to our dedicated listeners. This is your weather update, brought to you by the Philippine Atmospheric, Geophysical and Astronomical Services Administration, or PAGASA. We are issuing this advisory for Tropical Storm **Malakas**.

At this time, the PAGASA center has determined the location of **Malakas** as of four o’clock this morning. The center of the storm is situated approximately two point zero kilometers to the east of Mindanao. Its maximum sustained winds are reported at seventy-five kilometers per hour, with gustiness reaching up to ninety kilometers per hour, and a central pressure of nine hundred ninety-six hectopascals.

The storm is currently tracking in a general west-northwest direction, moving at a pace of fifteen kilometers per hour. While **Malakas** continues to maintain strength, it is crucial for all residents to remain vigilant and monitor official advisories. For continued safety, residents are advised to remain prepared for potential weather changes and to keep a close watch for any hoisting of a warning signal, including the possibility of a **Signal Number Two** in coastal areas, as conditions change.

## Forecast Track

Moving on to the forecast track for **Malakas**, the overall path shows a systematic movement eastward, remaining predominantly outside the Philippine Area of Responsibility.

Looking ahead, the modeling indicates that the storm will steadily progress over the next several days. By the time the forecast reaches the next day, on the tenth of April, **Malakas** is projected to be even further east. By the time we reach the eleventh of April, the storm is expected to be situated over an area far to the east of the Visayas region.

The forecast shows a significant intensification trend for the storm. By the twelfth of April, the intensity is projected to climb substantially. And as the storm continues its journey, by the thirteenth of April, the forecast indicates that the maximum sustained winds could reach one hundred fifty kilometers per hour, classifying it at its peak projected strength. This trajectory continues through the next two days, suggesting that the system will maintain or approach this powerful intensity level as it moves toward the east of Northern Luzon on the fifteenth of April.

It is important to understand that the entire projected track places **Malakas** hundreds of kilometers away from the major landmasses of the Philippines, generally moving toward the central Pacific.

## Affected Areas and Public Safety Advisory

Because **Malakas** is tracking well outside the Philippine Area of Responsibility, the immediate threat level to our islands remains low, however, extreme weather systems can generate hazardous conditions even hundreds of kilometers away.

Residents are urged to continue preparing for any adverse weather that might be associated with large tropical systems in the general vicinity. This includes preparing for potential changes in sea states for those in coastal or fishing communities.

We must repeat this advisory: while **Malakas** is forecast to remain offshore, we urge all residents to remain alert. Keep all emergency kits ready, secure loose objects around your homes, and ensure that all family members know the designated evacuation routes, should local authorities require them.

Furthermore, we must emphasize that PAGASA continuously monitors the storm's movement. We repeat that **Tropical Storm Malakas** is the system to monitor. And while the current forecast does not warrant a higher alert, residents should still be prepared for possible local advisories, so please stay tuned for updates regarding **Signal Number Two** or any other cautionary warnings.

## Closing

To summarize, while **Malakas** is projected to remain offshore and continue its strengthening path eastward, continuous vigilance and preparedness at the community level remain our highest priority.

We encourage everyone to rely only on official sources like PAGASA for your weather information.

This concludes our weather bulletin. We will be providing the next comprehensive update on this developing situation in exactly three hours, at the hour of the noon bulletin. Thank you for listening, and please stay safe.

## Tagalog Version

**Language:** Tagalog  │  **Words:** 684  │  **Est. reading time:** 4.6 min

---

# TROPICAL STORM MALAKAS: Weather Bulletin – Update for April Nine, Two Thousand Twenty-Two

*(Magsisimula ang boses sa isang malalim, kalmado, at awtoritatibong tono. Dapat ay may pagitan sa pagitan ng mga pangungusap para sa natural na paghinga ng radio broadcaster.)*

Magandang umaga po sa lahat ng nakikinig sa ating broadcast. Ito po ay isang paalala mula sa **PAGASA**, o Philippine Atmospheric, Geophysical, and Astronomical Services Administration. Patuloy po tayong magpapaalala sa ating komunidad tungkol sa pagsubaybay sa mga pagbabago ng panahon na dulot ng mga tropikal na bagyo.

***

## Kasalukuyang Sitwasyon

Sa kasalukuyan, patuloy po nating sinusubaybayan ang **Tropical Storm Malakas**. Ayon sa pinakahuling pagtatasa ng PAGASA, ang sentro po ng **Tropical Storm Malakas** ay matatagpo sa isang lokasyon na nasa labas po ng Philippine Area of Responsibility, o PAR.

Noong alas kuwatro ng umaga ngayong araw, ang sentro ng bagyo ay tinatayang nasa dalawang kilometro, silangan ng Mindanao. Ang lakas po ng hangin na naitala ay may maximum sustained winds na pitumpu't limang kilometro bawat oras, na may mga pagbugso na umaabot sa siyamnapung kilometro bawat oras.

Patungkol naman po sa paggalaw, ang **Tropical Storm Malakas** ay gumagalaw sa direksyong *west northwest* sa bilis na labinlima kilometro bawat oras.

***

## Forecast Track at Pag-uunlad

Patungo naman po tayo sa mas mahabang pagtataya, o forecast track ng **Tropical Storm Malakas**.

Ang pag-aaral ng mga meteorologist ay nagpapakita na habang nananatili itong labas ng ating bansa, ang **Tropical Storm Malakas** ay inaasahang magpapatuloy sa paglakad sa direksyong *west northwest*.

Ito po ay isang mahalagang babala: habang ang bagyo ay patuloy na gumagalaw patungong silangan ng ating mga isla, may nakasaad pong paglakas lalo pa ng bagyo.

Ayon sa forecast, sa loob ng mga darating na araw, inaasahan na aabot ang lakas ng hangin ng **Tropical Storm Malakas** sa antas ng Tropical Storm, at posibleng maging Tropical Cyclone, o bagyo. Sa loob ng mga darating na ika-animnapung oras, inaasahang aabot ang bilis ng hangin sa isang daan at limampu kilometro bawat oras, at posibleng mas mataas pa.

Ito pong paglakas ay nagpapahiwatig na kahit na malayo pa sa ating mga lugar, ang kalakasan ng bagyo ay patuloy na nagbabanta sa kalagayan ng karagatan at hangin.

***

## Mga Apektadong Lugar at Babala sa Karagatan

Sa ngayon po, dahil nananatili pa rin ang **Tropical Storm Malakas** sa labas ng PAR, ang pangunahing babala ay nakatuon sa **mga malalakas na alon at pagtaas ng tubig sa karagatan**.

Hinihikayat po ang lahat ng mga mangingisda at mga residente sa baybayin na maging lubhang maingat. Dahil sa paglalakbay ng **Tropical Storm Malakas**, inaasahan pong magkakaroon ng malalakas na hangin, matatayog na alon, at posibleng pagbaha sa ilang mababang lugar.

Dapat po nating tandaan na kahit walang direktang pagtama sa lupa, ang pagbabago ng panahon ay maaaring magdulot ng pagbaha, at ang mga paggalaw ng karagatan ay maaaring maging mapanganib.

***

## Payo sa Kaligtasan at Paghahanda

Bilang paghahanda sa anumang posibleng pagbabago ng panahon, muli po nating ipinapaalala ang mga sumusunod na hakbang:

Una, pangalagaan po ang ating mga ari-arian. Ipatong o itanim po sa matibay na lugar ang anumang mga bagay na maaaring mailipat ng malakas na hangin, tulad po ng mga paso, gamit sa labas, at mga bubong na hindi matibay.

Pangalawa, para sa mga residente sa mababang lugar, mangyaring maghanda ng mga emergency kit: kabilang na po ang inuming tubig, mga de-latang pagkain, flashlight, at mga pangunahing gamot.

Pangatlo, huwag po tayong magpabaya sa pagpapatakbo ng mga kasangkapan at kuryente nang walang kinakailangang kaalaman. Sundin po ninyo ang mga patakaran ng inyong lokal na pamahalaan.

Ang pag-uulat po ng PAGASA ay patuloy na nasa inyo. Huwag po kayong magpapadala sa mga haka-haka; ang impormasyon ay kailangang galing sa mga awtoridad.

***

## Pangwakas

Bilang pagtatapos, mga kababayan, paalalahanan po namin kayong manatiling alerto at patuloy na mag-monitor ng mga update tungkol sa **Tropical Storm Malakas**. Ang pag-iingat ay hindi lang po isang aksyon, kundi isang pang-araw-araw na paghahanda.

Ito po ang inyong weather bulletin mula sa PAGASA. Maraming salamat po sa pakikinig. Ang susunod po nating scheduled advisory ay muling ipapahayag sa inyo sa ganap na alas-onse ng umaga. Manatiling kalmado, at mag-ingat po kayo.

## Cebuano Version

**Language:** Cebuano  │  **Words:** 663  │  **Est. reading time:** 4.4 min

---

# 🌬️ Tropical Storm "Malakas" Weather Update: Report sa Pag-uyog sa Kalasangan

**(Sound Effect: Short, upbeat, yet serious jingle)**

**Radio Broadcaster (Warm, friendly, energetic tone):**

Maayong buntag, mga kauban! Kumusta mo diha sa inyong mga balay ug lugar? Kami ang inyong kauban niining mga pahibalo sa panahon. Sa pagkaandam nato sa mga pagbag-o sa klima, importante gyud nga atong mahibalo ang gibug-aton sa kahimtang sa dagat ug sa mga bagyo.

Ato ning hisgutan karon, usa ka update gikan sa PAGASA—ang atong Philippine Atmospheric, Geophysical and Astronomical Services Administration. Kining advisory mahitungod sa Tropical Storm nga gitudlo og **MALAKAS**.

**Mga kauban, paghinumdom aron kanunay kitang nag-amping sa panahon. Importante gyud ni sa atong pagkinabuhi. Kaya nato ni, basta magkahiusa ta ug mag-andam!**

***

### 🌪️ Unsay Nahitabo Karon ang Tropical Storm Malakas?

Karon, sa atong pagka-istorbo-istorbo sa pagka-balita, mao ni ang sitwasyon ni **Malakas**.

Base sa latest report, ang sentro sa **Tropical Storm Malakas** karon sa alas singko sa pagka-buntag, gitan-aw nga naa pa gihapon kini *sa gawas* sa atong Philippine Area of Responsibility—meaning, wala pa gyud kaayo kitang kaatbang direkta sa kasingkasing niining bagyo.

Pero, dili lang to nagpasalo. Ang **Malakas** dili gyud nagpahuway! Ang iyang kinaiyahan kay nag-ig-init (intensifying), mga kauban. Sa pagka-buntag, ang iyang kusog kay nagpabilin sa pito ug katingog ka kilometro (75 km/h) ang hangin nga kasagaran, pero nagdala pud ni og mga gustiness nga mahatag sa siyam ug katingog ka kilometro (90 km/h). Ang mga wind gust kay gamay ra nga kaagi, pero grabe gihapon!

Ang direksyon niya kay naglihok og West-Northwest, sa kalami lang nga og napulo ug singko ka kilometro (15 km/h).

***

### 🗺️ Asa Padulong ang Bagyo? Ang Pag-analisa sa Pag-uswag

Karon ha, atong tan-awon ang pag-agi sa **Malakas** sa mga mga adlaw nga moabot pa. Kini nga forecast kay makatabang gyud nato nga mag-andam og maayo.

Ang mapa nga gitanyag sa PAGASA nagpakita nga si **Malakas** mo-uswag pag-ayo. Sa unang paglabay, mo-lingaw pa kini sa pagka-kanunay, pero sa paglabay sa mga alas singko sa kaadlawon sa sunod semana, mao na ang atong bantayan.

Ang pag-forecast nagpakita nga nag-uswag ni **Malakas** gikan sa Tropical Storm paingon paingon sa pagka-Typhoon. Kini nagpasabot nga magkadaghan ang kusog sa hangin nga iyang dala.

Ang mga pagbanaw kay nagpaila nga ang **Malakas** mag-agi pa sa kasikbit nga mga lugar sa Visayas ug sa Luzon, pero, mga kauban, daghang babala nga **nagpabilin pa gihapon siya sa gawas sa atong Official Area**. Busa, bisan pa sa pagka-Typhoon, ang pag-monitor sa hangin, ulan, ug dagat kay importante pa gihapon.

***

### ⚠️ Kinsa ang Apektado ug Unsa ang Imong Kinahanglan Buhaton?

Tungod kay ang **Malakas** kay grabe ka kusog, bisan pa nga malayo pa siya sa atong kasingkasing, dili nato i-sira ang atong pag-amping.

Ang kasagarang apektado sa mga bagyo mao ang mga komunidad nga sa baybayon, ang mga lugar nga permi kaayo mo-tawa sa hangin, ug ang mga mga panginda rahon.

**Importante nga mga pahinumdum, mga kauban:**

1.  **Monitor:** Magpabilin kamo nga nag-monitor sa mga update gikan sa PAGASA. Dili pwede mag-undang ang pagka-andam.
2.  **Pag-andam og Emergency Kit:** Siguraduhon nga naa kamo'y emergency kit. Ilakip niini ang first-aid supplies, flashlight, extra batteries, ug daghang tubig.
3.  **Water Level:** Kung aduna moy mga mga lugar nga permi kaayo mo-tawa sa tubig-baha, pag-andam mo sa pag-evacuate kung mag-utos ang lokal nga kagamhanan.
4.  **Pamilya:** Pag-istorya mo sa inyong pamilya. Kanang mga importante nga contact number, isulat ninyo sa lugar nga dali tan-awon.

***

### 📻 Pangwakas ug Pagpangandam

Mga kauban, kinahanglan gyud ta nga magpabilin nga kalmado, pero magpabilin sad kitang pag-andam. Ang pag-monitor sa mga grabe nga bagyo, sama ni **Malakas**, kay usa ka *ongoing process*.

Dili nako kamo pasagdan nga maghilak og kasikbit nga kasikbit sa pagka-andam. Mag-andam ta og maayo. Ang pagka-konektado ug pagka-informado mao ang atong pinakadako nga proteksyon.

Mga kauban, atong atimanon pag-ayo ang atong mga balay ug ang atong komunidad.

**Unya ha, atong tan-awon pag-usab ang mas detailed nga pag-update sa pagka-adlaw. Paglingawon og pagka-andam. Daghang salamat sa pagpaminaw!**

**(Sound Effect: Upbeat, reassuring jingle)**

## 5. Length Check

Verify word counts across all three languages are close to the 750-word target. If any are significantly short or long, we can adjust the prompts.

In [10]:
TARGET_WORDS = 750
WORDS_PER_MIN = 150

print("\nLENGTH SUMMARY")
print("=" * 60)
print(f"{'Bulletin':<35} {'Lang':>4} {'Words':>6}  {'Minutes':>7}  {'vs Target':>10}")
print("-" * 60)

for result in results:
    diff = result['word_count'] - TARGET_WORDS
    diff_str = f"+{diff}" if diff >= 0 else str(diff)
    label = result['stem'].replace('PAGASA_', '')
    lang = result['language'].upper()
    print(f"{label:<35} {lang:>4} {result['word_count']:>6}  {result['reading_minutes']:>6.1f}m  {diff_str:>10}")

print("-" * 60)
print(f"Target: {TARGET_WORDS} words ≈ {TARGET_WORDS/WORDS_PER_MIN:.0f} minutes at {WORDS_PER_MIN} wpm")
print(f"\n✓ {len(results)} scripts saved to: {output_dir.absolute()}")


LENGTH SUMMARY
Bulletin                            Lang  Words  Minutes   vs Target
------------------------------------------------------------
20-19W_Pepito_SWB#01                  EN    676     4.5m         -74
20-19W_Pepito_SWB#01                  TL    672     4.5m         -78
20-19W_Pepito_SWB#01                 CEB    644     4.3m        -106
22-TC02_Basyang_TCA#01                EN    654     4.4m         -96
22-TC02_Basyang_TCA#01                TL    684     4.6m         -66
22-TC02_Basyang_TCA#01               CEB    663     4.4m         -87
------------------------------------------------------------
Target: 750 words ≈ 5 minutes at 150 wpm

✓ 6 scripts saved to: /Users/josereyes/Dev/gemma4-hackathon/notebooks/../data/radio_bulletins
